In [ ]:
%pip install numpy pandas numba scipy pyarrow huggingface-hub requests plotly

In [ ]:
import os
import sys

REPO_URL  = "https://github.com/payamdav/pycrypto2.git"
REPO_NAME = "pycrypto2"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

REPO_PATH = os.path.abspath(REPO_NAME)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

In [ ]:
# Base asset — change here to switch asset for all cells below
asset = "btcusdt"

In [ ]:
import time
from packages.tools.candle_preloader import preload_candles
from packages.tools.candle_cache import preload_asset_candles

t0 = time.perf_counter()

# File cache (download if missing, then read from local parquet)
preload_candles([asset])

# In-memory cache (fast repeated slicing)
preload_asset_candles(asset)

t_preload = time.perf_counter() - t0
print(f"\nCandle pre-load total: {t_preload:.3f}s")

In [ ]:
from packages.tools.metrics_cache import (
    create_metrics_cache_base_file,
    metrics_cache_volume_median_iqr,
    metrics_cache_volume_mean_stddev,
)

t0 = time.perf_counter()

_t = time.perf_counter()
create_metrics_cache_base_file(asset)
print(f"  create_metrics_cache_base_file   {time.perf_counter() - _t:.3f}s")

_t = time.perf_counter()
metrics_cache_volume_median_iqr(asset)
print(f"  metrics_cache_volume_median_iqr  {time.perf_counter() - _t:.3f}s")

_t = time.perf_counter()
metrics_cache_volume_mean_stddev(asset)
print(f"  metrics_cache_volume_mean_stddev {time.perf_counter() - _t:.3f}s")

t_metrics = time.perf_counter() - t0
print(f"\nMetrics cache total: {t_metrics:.3f}s")

In [ ]:
print(f"Total pre-load + metrics time: {t_preload + t_metrics:.3f}s")

In [ ]:
# ── Parameters (edit here) ──────────────────────────────────
look_back          = 1440          # candles in the look-back window (includes anchor)
look_ahead         = 240           # candles in the look-ahead window
k                  = 100.0         # price scale factor (ratio 0.01 → 1.0)
bins_count         = 200           # KDE histogram bins over [-1, 1]
bandwidth          = 5             # kernel half-width in bins
kernel_type        = "Triangular"  # "Triangular" | "Epanechnikov" | "Uniform"
kde_ignore_borders = True          # exclude clipped prices (±1.0) from histogram
rel_height         = 0.5           # relative height for peak-width measurement
top_identifier     = "prominence"  # "prominence" | "height" — peak ranking key
peak_numbers       = 3             # number of peaks above and below
minimum_peak_score = 0.0           # min robust z-score a peak must reach to survive
dt_str             = "2025-12-12 20:00:00"   # target anchor minute (UTC)
# ────────────────────────────────────────────────────────────

from strategies.lbla_n_vp.lbla_n_vp import lookback_lookahead_normalized_vp

data = lookback_lookahead_normalized_vp(
    asset=asset, look_back=look_back, look_ahead=look_ahead,
    datetime=dt_str, k=k, bins_count=bins_count,
    bandwidth=bandwidth, kernel_type=kernel_type,
    kde_ignore_borders=kde_ignore_borders,
    rel_height=rel_height, top_identifier=top_identifier,
    peak_numbers=peak_numbers, minimum_peak_score=minimum_peak_score,
)

In [ ]:
# ── 3D Kalman chart (edit knobs here) ────────────────────────
import numpy as np

measurement_variance = 1.0                  # filter R
process_noise        = np.eye(3) * 0.03     # filter Q (3x3 eye matrix)
# ────────────────────────────────────────────────────────────

from strategies.lbla_n_vp.lbla_n_vp_chart import draw_chart_vp_continued_width_kalman_3d

draw_chart_vp_continued_width_kalman_3d(
    data, measurement_variance=measurement_variance, process_noise=process_noise
)